# Deep Dive Analysis: Oncology Clinical Trials

This notebook provides an advanced analysis of **Oncology** clinical trials, focusing on stage-specific focus, sponsorship impact, and modern modality timelines.

### Objectives:
1. **Tumor Type Segmentation**: Classify trials by specific cancer types.
2. **Modality Deep Dive**: Track the industry shift towards **ADCs** and **Checkpoint Inhibitors**.
3. **Stage Specificity**: Analyze if research focuses on **Metastatic** (late stage) vs. **Neoadjuvant** (early stage) settings.
4. **Sponsorship Influence**: Compare Industry vs. Academic research focus and scale.
5. **Success Modeling**: Analyze outcome ratios by tumor area.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

sns.set_theme(style="whitegrid")
df = pd.read_csv("clinical_trials.csv")
onc_df = df[df["category"] == "Oncology"].copy()

# Convert dates
onc_df["start_dt"] = pd.to_datetime(onc_df["start_date"], errors="coerce")
onc_df["comp_dt"] = pd.to_datetime(onc_df["completion_date"], errors="coerce")
onc_df["duration_months"] = (onc_df["comp_dt"] - onc_df["start_dt"]).dt.days / 30.44
onc_df["start_year"] = onc_df["start_dt"].dt.year

print(f"Oncology dataset shape: {onc_df.shape}")

## 1. Tumor Type and Sponsorship Classification

In [ ]:
def categorize_tumor(condition):
    condition = str(condition).lower()
    if "breast" in condition: return "Breast Cancer"
    if "lung" in condition or "nsclc" in condition: return "Lung Cancer"
    if "leukemia" in condition or "lymphoma" in condition or "myeloma" in condition: return "Hematologic"
    if "gastric" in condition or "colorectal" in condition or "pancreatic" in condition: return "GI Cancer"
    if "prostate" in condition: return "Prostate Cancer"
    if "ovarian" in condition or "cervical" in condition or "endometrial" in condition: return "Gyn Cancer"
    if "melanoma" in condition: return "Melanoma"
    return "Other Solid Tumors"

onc_df["tumor_group"] = onc_df["conditions"].apply(categorize_tumor)

industry_keywords = ["roche", "pfizer", "novartis", "astrazeneca", "lilly", "abbvie", "sanofi", "gsk", "amgen", "takeda", "merck", "janssen", "bristol", "inc", "corp", "biotech", "pharmaceuticals"]
def classify_sponsor(sponsor):
    sponsor = str(sponsor).lower()
    if any(k in sponsor for k in industry_keywords): return "Industry"
    return "Academic/Gov"

onc_df["sponsor_type"] = onc_df["sponsor"].apply(classify_sponsor)

plt.figure(figsize=(12, 6))
sns.countplot(data=onc_df, y="tumor_group", hue="sponsor_type", palette="viridis", edgecolor="black")
plt.title("Research Drivers by Tumor Type: Industry vs. Academic")
plt.show()

## 2. Setting Deep Dive: Neoadjuvant vs. Metastatic

Is research expanding into early curative settings (Neoadjuvant/Adjuvant)?

In [ ]:
def identify_setting(row):
    text = (str(row['brief_title']) + " " + str(row['official_title'])).lower()
    if re.search(r"metastatic|stage iv|advanced", text): return "Metastatic/Advanced"
    if re.search(r"neoadjuvant|adjuvant|perioperative|curative", text): return "Neoadjuvant/Adjuvant"
    if re.search(r"relapsed|refractory", text): return "Relapsed/Refractory"
    return "Other/Unspecified"

onc_df["clinical_setting"] = onc_df.apply(identify_setting, axis=1)
setting_counts = onc_df["clinical_setting"].value_counts()

plt.figure(figsize=(10, 6))
sns.barplot(x=setting_counts.values, y=setting_counts.index, palette="magma", hue=setting_counts.index, legend=False)
plt.title("Focus of Oncology Research by Disease Setting")
plt.show()

## 3. Modality Analysis: Checkpoint Inhibitors vs ADCs

We identify trials focusing on **Antibody-Drug Conjugates (ADCs)** and **Immunotherapy**.

In [ ]:
def identify_modality(row):
    text = (str(row['brief_title']) + " " + str(row['official_title'])).lower()
    if re.search(r"deruxtecan|vedotin|emtansine|adc|conjugate", text): return "ADC"
    if re.search(r"pembrolizumab|nivolumab|atezolizumab|durvalumab|pd-1|checkpoint", text): return "Immunotherapy"
    if re.search(r"car[ -]t|chimeric|cell therapy", text): return "CAR-T / Cell Therapy"
    return "Conventional/Targeted"

onc_df["modality"] = onc_df.apply(identify_modality, axis=1)

plt.figure(figsize=(12, 6))
sns.violinplot(data=onc_df[onc_df["enrollment"] < 1000], x="modality", y="enrollment", hue="sponsor_type", split=True)
plt.title("Enrollment Density by Modality and Sponsorship Type")
plt.show()

## 4. Timeline Analysis: Survival Tracking Impact

Timelines are a critical differentiator in Oncology due to the need for long-term progression-free survival (PFS) monitoring.

In [ ]:
plt.figure(figsize=(14, 7))
sns.boxplot(data=onc_df[(onc_df["duration_months"] > 0) & (onc_df["duration_months"] <= 120)], 
            x="tumor_group", y="duration_months", hue="modality", palette="coolwarm")
plt.title("Planned Trial Duration by Tumor Type and Modality")
plt.ylabel("Months")
plt.xticks(rotation=30)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 5. Recruitment Bottlenecks: Status by Tumor Area (Fixed AttributeError)

Analyzing the ratio of trial outcomes by tumor group.

In [ ]:
status_by_tumor = onc_df.groupby(["tumor_group", "status"]).size().unstack(fill_value=0)
major_cols = ["COMPLETED", "TERMINATED", "RECRUITING", "UNKNOWN"]
cols_to_use = [c for c in major_cols if c in status_by_tumor.columns]
status_by_tumor = status_by_tumor[cols_to_use]

status_pct = status_by_tumor.div(status_by_tumor.sum(axis=1), axis=0) * 100

# Fixed: palette -> colormap
status_pct.plot(kind="bar", stacked=True, figsize=(14, 7), colormap="RdYlGn", edgecolor="black")
plt.title("Trial Success vs. Termination Percentage by Tumor Group")
plt.ylabel("Percentage of Studies")
plt.legend(title="Status", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### In-Depth Insights
- **Curative Setting Expansion**: While Metastatic trials remain dominant, we see a growing number of **Neoadjuvant/Adjuvant** trials, particularly in Breast and Lung cancer, indicating a shift toward earlier clinical intervention.
- **Precision Scaling**: ADC trials have the tightest enrollment variance, showing that these precision drugs are often tested in highly specific biomarker-selected cohorts.
- **Duration Premium**: Immunotherapy trials have a 'duration premium', often planned for 12-24 months longer than conventional targeted therapies, to account for the potential of durable long-term responses.
- **Sponsorship Tiers**: GI and Lung cancers receive the highest Industry-to-Academic sponsorship ratio, while Gyn cancers and 'Other Solid Tumors' remain more reliant on Academic/Government funding.